<a href="https://colab.research.google.com/github/mironovromandenisovich-creator/tetstst/blob/main/notebooks/comfyui_colab_with_manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [ ]:
# ==========================================
# 📦 0. УСТАНОВКА БАЗОВЫХ ПАКЕТОВ И УТИЛИТ
# ==========================================
print("📦 [1/5] Установка зависимостей и Cloudflared...")
!pip install -q GitPython sageattention
!apt-get install -y -qq aria2 > /dev/null 2>&1

# Скачиваем официальный бинарник Cloudflared для туннеля
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import os
import re
import time
import socket
import threading
import subprocess

PORT = 8188

# ==========================================
# 🚀 1. ПОДГОТОВКА ЯДРА COMFYUI И НОД
# ==========================================
print("\n🚀 [2/5] Проверка ядра ComfyUI и установка видео-нод...")

# ComfyUI Core
if not os.path.exists("/content/ComfyUI/main.py"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
else:
    %cd /content/ComfyUI
    !git pull

# ComfyUI-Manager
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
else:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !git pull

# VideoHelperSuite (Критично для сборки MP4 со звуком в MiniMax H3)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite"):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite

# Установка зависимостей ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# ==========================================
# 📁 2. СОЗДАНИЕ СТРУКТУРЫ ПАПОК
# ==========================================
print("\n📁 [3/5] Создание рабочих папок для моделей MiniMax H3...")
os.makedirs("/content/ComfyUI/models/diffusion_models", exist_ok=True)
os.makedirs("/content/ComfyUI/models/text_encoders", exist_ok=True)
os.makedirs("/content/ComfyUI/models/vae", exist_ok=True)

# ==========================================
# 📥 3. СКАЧИВАНИЕ ТОЛЬКО НЕОБХОДИМЫХ МОДЕЛЕЙ ДЛЯ H3
# ==========================================
print("\n📥 [4/5] Загрузка официального набора MiniMax H3 (T2V, I2V, R2V)...")

HF_BASE = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"

# 1. Текстовый энкодер Qwen3-VL 32B (Универсален для T2V, I2V и R2V) ~15.7 ГБ
print("  -> [1/5] Скачиваем текстовый энкодер Qwen3-VL 32B...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "{HF_BASE}/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors" \
  -d /content/ComfyUI/models/text_encoders/ -o qwen3vl_32b_minimax_h3_int8_convrot.safetensors

# 2. Модель диффузии FL2VA (Чистый BF16 для Text-to-Video и Image-to-Video) ~40.2 ГБ
print("  -> [2/5] Скачиваем модель для T2V и I2V (FL2VA BF16)...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "{HF_BASE}/diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors" \
  -d /content/ComfyUI/models/diffusion_models/ -o minimax_h3_fl2va_pruned_bf16.safetensors

# 3. Модель диффузии Ref2VA (Чистый BF16 для Reference-to-Video) ~40.2 ГБ
print("  -> [3/5] Скачиваем модель для R2V (Ref2VA BF16)...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "{HF_BASE}/diffusion_models/minimax_h3_ref2va_pruned_bf16.safetensors" \
  -d /content/ComfyUI/models/diffusion_models/ -o minimax_h3_ref2va_pruned_bf16.safetensors

# 4. Видеодекодер VAE (Студийный FP16 без сжатия) ~5.2 ГБ
print("  -> [4/5] Скачиваем видеодекодер Video VAE FP16...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "{HF_BASE}/vae/minimax_h3_video_vae_fp16.safetensors" \
  -d /content/ComfyUI/models/vae/ -o minimax_h3_video_vae_fp16.safetensors

# 5. Аудиодекодер VAE (FP32 для генерации 32 кГц звука и речи) ~0.6 ГБ
print("  -> [5/5] Скачиваем аудиодекодер Audio VAE FP32...")
!aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M \
  "{HF_BASE}/vae/minimax_h3_audio_vae_fp32.safetensors" \
  -d /content/ComfyUI/models/vae/ -o minimax_h3_audio_vae_fp32.safetensors

# ==========================================
# 🌐 4. ЗАПУСК CLOUDFLARE И COMFYUI
# ==========================================
print("\n🌐 [5/5] Настройка туннеля и старт ComfyUI на A100...")
!pkill -f "main.py"
!pkill -f "cloudflared"

def wait_for_port_and_launch(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n" + "=" * 65)
    print("✅ СЕРВЕР COMFYUI ЗАПУЩЕН НА A100! ПОДКЛЮЧАЕМ ТУННЕЛЬ...")

    log_path = "/content/cloudflared.log"
    if os.path.exists(log_path):
        os.remove(log_path)

    cmd = f"cloudflared tunnel --url http://127.0.0.1:{port} --protocol http2 > {log_path} 2>&1"
    subprocess.Popen(cmd, shell=True)

    url = None
    for _ in range(40):
        time.sleep(1)
        if os.path.exists(log_path):
            with open(log_path, "r") as f:
                content = f.read()
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
                if match:
                    url = match.group(0)
                    break

    if url:
        print(f"👉 ВАША ССЫЛКА НА ИНТЕРФЕЙС: {url}")
    else:
        print("⚠️ Не удалось получить ссылку автоматически. Проверьте лог: !cat /content/cloudflared.log")
    print("=" * 65 + "\n")

threading.Thread(target=wait_for_port_and_launch, daemon=True, args=(PORT,)).start()

# Запуск с флагом --highvram (запрещает ненужный сброс модели в системную RAM)
%cd /content/ComfyUI
!python main.py --listen 127.0.0.1 --port 8188 --enable-cors-header --dont-print-server --highvram